In [1]:
import pandas as pd
from getpass import getpass
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# 1. Load the clean data
loans = pd.read_parquet("../data/processed/loans_clean.parquet")
print("Loaded:", loans.shape)

# 2. Ask for the PostgreSQL password (so it's never written in the code)
password = getpass("Enter your PostgreSQL password: ")

# 3. Create the connection to the loan_risk database
db_url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password=password,
    host="localhost",
    port=5432,
    database="loan_risk",
)
engine = create_engine(db_url)

# 4. Test the connection
with engine.connect() as conn:
    print("Connected to:", conn.execute(text("SELECT version()")).scalar())

Loaded: (1322229, 49)
Connected to: PostgreSQL 18.6 on x86_64-windows, compiled by msvc-19.44.35229, 64-bit


In [2]:
import csv
import time
from io import StringIO

# Fast bulk loader: uses PostgreSQL's COPY command instead of row-by-row inserts
def psql_insert_copy(table, conn, keys, data_iter):
    dbapi_conn = conn.connection
    with dbapi_conn.cursor() as cur:
        buffer = StringIO()
        csv.writer(buffer).writerows(data_iter)
        buffer.seek(0)
        columns = ", ".join(f'"{k}"' for k in keys)
        table_name = f"{table.schema}.{table.name}" if table.schema else table.name
        cur.copy_expert(f"COPY {table_name} ({columns}) FROM STDIN WITH CSV", buffer)

# Load the data in batches of 1 lakh rows (keeps memory low)
start = time.time()
loans.to_sql(
    "loans",
    engine,
    if_exists="replace",
    index=False,
    chunksize=100_000,
    method=psql_insert_copy,
)
print("Load time:", round(time.time() - start, 1), "seconds")

# Verify the row count inside PostgreSQL
with engine.connect() as conn:
    row_count = conn.execute(text("SELECT COUNT(*) FROM loans")).scalar()
print("Rows in PostgreSQL:", row_count)
print("Rows in Python    :", len(loans))
print("Match:", row_count == len(loans))

Load time: 151.6 seconds
Rows in PostgreSQL: 1322229
Rows in Python    : 1322229
Match: True
